# Dataset Preview (HaluEval)

## Goals
1. Verify that the processed splits load correctly (`train/val/test`).
2. Show a simple “before/after” view: separate `prompt/response` vs. a combined `text` field.
3. Inspect basic distributions: `label`, `task`, and text lengths.
4. Display a few random examples of hallucinations vs. non-hallucinations.

> This notebook is intentionally lightweight and visual. It does not train models.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 140)

# --- Repo bootstrap (running from notebooks/) ---
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_splits import load_splits
from src.utils.experiment import seed_everything

# Reproducibility
SEED = 42
seed_everything(SEED)

# Output directory
REPORTS_DIR = ROOT / "reports"
OUTPUT_DIR = REPORTS_DIR / "nb01_dataset_preview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

train_df, val_df, test_df = load_splits(root=ROOT)

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

# Use a stable sample for faster visuals
sample = train_df.sample(2000, random_state=SEED).copy()
sample.head(3)

In [2]:
# BEFORE: prompt/response are stored separately (useful for feature engineering)
sample[["task", "label", "prompt", "response"]].head(2)

# AFTER: create a combined text field (useful for TF-IDF baselines)
sample = sample.copy()
sample["text"] = sample["prompt"].fillna("").astype(str) + "\n\n" + sample["response"].fillna("").astype(str)

sample[["task", "label", "text"]].head(2)


,task,label,text
13005,dialogue,1,[Human]: Do you like Will Smith? [Assistant]: Absolutely! I love The Pursuit of Happyness. Have you seen it? [Human]: Maybe did he kill ...
36568,summarization,0,Three people have been killed after clubbers stampeded into a nightclub to see a British punk band called Doom. The crowd tried to storm...


## Label distribution (Train)

In [ ]:
label_counts = train_df["label"].value_counts().sort_index()
print("Train label counts:", label_counts.to_dict())

# Save distribution
label_df = label_counts.reset_index()
label_df.columns = ["label", "count"]
label_df.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False)

label_counts.plot(kind="bar")
plt.title("Label distribution (Train) (0=non-hallucination, 1=hallucination)")
plt.xlabel("label")
plt.ylabel("count")
plt.savefig(PLOTS_DIR / "label_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

## Task distribution (Train)

In [ ]:
task_counts = train_df["task"].value_counts()
print("Train task counts (top 20):", task_counts.head(20).to_dict())

# Save distribution
task_df = task_counts.reset_index()
task_df.columns = ["task", "count"]
task_df.to_csv(OUTPUT_DIR / "task_distribution.csv", index=False)

task_counts.plot(kind="bar")
plt.title("Task distribution (Train)")
plt.xlabel("task")
plt.ylabel("count")
plt.savefig(PLOTS_DIR / "task_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

## Length distributions (sample)

In [ ]:
df2 = sample.copy()
df2["prompt_len"] = df2["prompt"].fillna("").astype(str).str.len()
df2["response_len"] = df2["response"].fillna("").astype(str).str.len()

print(df2[["prompt_len", "response_len"]].describe())

df2["response_len"].plot(kind="hist", bins=50)
plt.title("Response length distribution (characters) — sample from train")
plt.xlabel("response_len")
plt.ylabel("count")
plt.savefig(PLOTS_DIR / "response_length_hist.png", dpi=100, bbox_inches="tight")
plt.show()

## Length vs. label (boxplot, sample)

In [ ]:
# Visual comparison: do hallucinations tend to be longer/shorter?
df2.boxplot(column="response_len", by="label")
plt.title("Response length by label — sample from train")
plt.suptitle("")  # remove pandas auto-title
plt.xlabel("label (0=non-hallucination, 1=hallucination)")
plt.ylabel("response_len")
plt.savefig(PLOTS_DIR / "response_length_by_label.png", dpi=100, bbox_inches="tight")
plt.show()

## Random examples (train)

In [ ]:
def show_examples(df: pd.DataFrame, label: int, n: int = 2, seed: int = 1, max_chars: int = 800) -> None:
    ex = df[df["label"] == label].sample(n, random_state=seed)
    for i, row in enumerate(ex.itertuples(index=False), 1):
        prompt = "" if pd.isna(row.prompt) else str(row.prompt)
        response = "" if pd.isna(row.response) else str(row.response)

        print("=" * 100)
        print(f"Example #{i} | task={row.task} | label={row.label}")
        print("- prompt:")
        print(prompt[:max_chars] + (" ..." if len(prompt) > max_chars else ""))
        print("\n- response:")
        print(response[:max_chars] + (" ..." if len(response) > max_chars else ""))

print("### Examples: Hallucination (label=1)")
show_examples(train_df, label=1, n=2, seed=SEED)

print("\n\n### Examples: Non-hallucination (label=0)")
show_examples(train_df, label=0, n=2, seed=SEED)

## Notes / Limitations
- Visualizations use a fixed-size sample for speed.
- Label and task distributions are computed on the full training split.
- This notebook is meant for sanity-checking the processed schema and basic dataset characteristics.
